In [1]:
from pathlib import Path
import sys

import polars as pl

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

from ufc_scraper.utils import read_significant_strikes


In [2]:
significant_strikes = read_significant_strikes().collect()

In [ ]:
significant_strikes

In [ ]:
import polars as pl

categories_values = ["sig_str", "head", "body", "leg", "distance", "clinch", "ground"]
categories_pct = ["sig_str_pct"]
fighters = [1, 2]

exprs_values = []
exprs_values_pct = []
exprs_pct = []

# all_categories = categories_values + categories_pct

all_categories = categories_values


cols_to_drop = [
    f"{category}_fighter{fighter}"
    for category in all_categories
    for fighter in fighters
]

for category in categories_values:
    for fighter in fighters:
        exprs_values.extend([
            pl.col(f"{category}_fighter{fighter}")
            .str.extract(r"(\d+) of (\d+)", 1)
            .cast(pl.Int64)
            .alias(f"{category}_landed_fighter{fighter}"),

            pl.col(f"{category}_fighter{fighter}")
            .str.extract(r"(\d+) of (\d+)", 2)
            .cast(pl.Int64)
            .alias(f"{category}_attempted_fighter{fighter}")
        ])

# for category in categories_pct:
#     for fighter in fighters:
#         exprs_pct.extend([
#             pl.col(f"{category}_fighter{fighter}")
#             .str.replace("%", "")
#             .cast(pl.Float64)
#             # .truediv(100)
#             .alias(f"{category}_fighter{fighter}")
#         ])

for category in categories_values:
    for fighter in fighters:
        exprs_values_pct.append(
            (
               100 * pl.col(f"{category}_landed_fighter{fighter}")
                / pl.col(f"{category}_attempted_fighter{fighter}")
            )
            .fill_nan(0)
            .fill_null(0)
            .round(2)
            .alias(f"{category}_pct_fighter{fighter}")
)

significant_strikes2 = significant_strikes.drop('sig_str_pct_fighter1', 'sig_str_pct_fighter2').with_columns(*exprs_values).with_columns(*exprs_values_pct).drop(cols_to_drop)

significant_strikes2 = significant_strikes2.with_columns(
    pl.col("round_label")
      .str.extract(r"(\d+)")
      .cast(pl.Int64)
      .alias("round"),
    
    pl.col("scraped_at")
      .str.strip_chars('"')
      .str.to_datetime(format="%Y-%m-%dT%H:%M:%S%z"),

    pl.col("ingested_at")
      .str.strip_chars('"')
      .str.to_datetime(format="%Y-%m-%dT%H:%M:%S%z"),
    
    pl.col("fighter1").str.strip_chars(),

    pl.col("fighter2").str.strip_chars()

).drop("round_label")

In [ ]:
significant_strikes2.columns

In [ ]:
len(significant_strikes2.columns)

In [ ]:
len(significant_strikes.columns)

In [ ]:
significant_strikes2

In [ ]:
significant_strikes

In [ ]:
fighter1_df = significant_strikes2.select(
    "fight_id",
    "event_id",
    "round",
    "scraped_at",
    "ingested_at",

    pl.col("fighter1").alias("fighter"),
    pl.col("fighter2").alias("opponent"),

    pl.col("sig_str_landed_fighter1").alias("sig_str_landed"),
    pl.col("sig_str_attempted_fighter1").alias("sig_str_attempted"),
    pl.col("sig_str_pct_fighter1").alias("sig_str_pct"),

    pl.col("head_landed_fighter1").alias("head_landed"),
    pl.col("head_attempted_fighter1").alias("head_attempted"),
    pl.col("head_pct_fighter1").alias("head_pct"),

    pl.col("body_landed_fighter1").alias("body_landed"),
    pl.col("body_attempted_fighter1").alias("body_attempted"),
    pl.col("body_pct_fighter1").alias("body_pct"),

    pl.col("leg_landed_fighter1").alias("leg_landed"),
    pl.col("leg_attempted_fighter1").alias("leg_attempted"),
    pl.col("leg_pct_fighter1").alias("leg_pct"),

    pl.col("distance_landed_fighter1").alias("distance_landed"),
    pl.col("distance_attempted_fighter1").alias("distance_attempted"),
    pl.col("distance_pct_fighter1").alias("distance_pct"),

    pl.col("clinch_landed_fighter1").alias("clinch_landed"),
    pl.col("clinch_attempted_fighter1").alias("clinch_attempted"),
    pl.col("clinch_pct_fighter1").alias("clinch_pct"),

    pl.col("ground_landed_fighter1").alias("ground_landed"),
    pl.col("ground_attempted_fighter1").alias("ground_attempted"),
    pl.col("ground_pct_fighter1").alias("ground_pct"),
)

fighter2_df = significant_strikes2.select(
    "fight_id",
    "event_id",
    "round",
    "scraped_at",
    "ingested_at",

    pl.col("fighter2").alias("fighter"),
    pl.col("fighter1").alias("opponent"),

    pl.col("sig_str_landed_fighter2").alias("sig_str_landed"),
    pl.col("sig_str_attempted_fighter2").alias("sig_str_attempted"),
    pl.col("sig_str_pct_fighter2").alias("sig_str_pct"),

    pl.col("head_landed_fighter2").alias("head_landed"),
    pl.col("head_attempted_fighter2").alias("head_attempted"),
    pl.col("head_pct_fighter2").alias("head_pct"),

    pl.col("body_landed_fighter2").alias("body_landed"),
    pl.col("body_attempted_fighter2").alias("body_attempted"),
    pl.col("body_pct_fighter2").alias("body_pct"),

    pl.col("leg_landed_fighter2").alias("leg_landed"),
    pl.col("leg_attempted_fighter2").alias("leg_attempted"),
    pl.col("leg_pct_fighter2").alias("leg_pct"),

    pl.col("distance_landed_fighter2").alias("distance_landed"),
    pl.col("distance_attempted_fighter2").alias("distance_attempted"),
    pl.col("distance_pct_fighter2").alias("distance_pct"),

    pl.col("clinch_landed_fighter2").alias("clinch_landed"),
    pl.col("clinch_attempted_fighter2").alias("clinch_attempted"),
    pl.col("clinch_pct_fighter2").alias("clinch_pct"),

    pl.col("ground_landed_fighter2").alias("ground_landed"),
    pl.col("ground_attempted_fighter2").alias("ground_attempted"),
    pl.col("ground_pct_fighter2").alias("ground_pct"),
)

significant_strikes_long = pl.concat(
    [fighter1_df, fighter2_df],
    how="vertical"
)

In [ ]:
significant_strikes_long

In [ ]:
import polars as pl

categories = ["sig_str", "head", "body", "leg", "distance", "clinch", "ground"]
metrics = ["landed", "attempted", "pct"]

base_cols = [
    "fight_id",
    "event_id",
    "round",
    "scraped_at",
    "ingested_at",
]

def build_fighter_view(df: pl.DataFrame, fighter_num: int) -> pl.DataFrame:
    other = 2 if fighter_num == 1 else 1

    exprs = [
        pl.col(f"fighter{fighter_num}").alias("fighter"),
        pl.col(f"fighter{other}").alias("opponent"),
    ]

    for category in categories:
        for metric in metrics:
            exprs.append(
                pl.col(f"{category}_{metric}_fighter{fighter_num}")
                .alias(f"{category}_{metric}")
            )

    return df.select(*base_cols, *exprs)

fighter1_df = build_fighter_view(significant_strikes2, 1)
fighter2_df = build_fighter_view(significant_strikes2, 2)

significant_strikes_long = pl.concat(
    [fighter1_df, fighter2_df],
    how="vertical",
)

In [ ]:
significant_strikes_long

### Pipeline Code

In [1]:
from pathlib import Path
import sys

import polars as pl

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

from ufc_scraper.utils import read_significant_strikes


In [2]:
significant_strikes = read_significant_strikes().collect()

In [5]:
significant_strikes.shape

(26, 23)

In [6]:
significant_strikes.head()

round_label,fighter1,sig_str_fighter1,sig_str_pct_fighter1,head_fighter1,body_fighter1,leg_fighter1,distance_fighter1,clinch_fighter1,ground_fighter1,fighter2,sig_str_fighter2,sig_str_pct_fighter2,head_fighter2,body_fighter2,leg_fighter2,distance_fighter2,clinch_fighter2,ground_fighter2,scraped_at,ingested_at,event_id,fight_id
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""Round 1""","""Anthony Hernandez""","""10 of 19""","""52%""","""10 of 19""","""0 of 0""","""0 of 0""","""5 of 12""","""5 of 7""","""0 of 0""","""Gregory Rodrigues""","""25 of 33""","""75%""","""15 of 21""","""10 of 12""","""0 of 0""","""14 of 21""","""9 of 10""","""2 of 2""","""2026-09-03T12:54:37+00:00""","""2026-09-03T13:23:47+00:00""","""a0a69dc9914ef6e1""","""fb77b08a90b92d5b"""
"""Round 2""","""Anthony Hernandez""","""19 of 39""","""48%""","""19 of 39""","""0 of 0""","""0 of 0""","""18 of 37""","""1 of 2""","""0 of 0""","""Gregory Rodrigues""","""52 of 83""","""62%""","""26 of 53""","""14 of 16""","""12 of 14""","""41 of 65""","""5 of 8""","""6 of 10""","""2026-09-03T12:54:37+00:00""","""2026-09-03T13:23:47+00:00""","""a0a69dc9914ef6e1""","""fb77b08a90b92d5b"""
"""Round 3""","""Anthony Hernandez""","""12 of 20""","""60%""","""12 of 20""","""0 of 0""","""0 of 0""","""4 of 12""","""5 of 5""","""3 of 3""","""Gregory Rodrigues""","""12 of 23""","""52%""","""10 of 19""","""1 of 3""","""1 of 1""","""5 of 12""","""7 of 11""","""0 of 0""","""2026-09-03T12:54:37+00:00""","""2026-09-03T13:23:47+00:00""","""a0a69dc9914ef6e1""","""fb77b08a90b92d5b"""
"""Round 4""","""Anthony Hernandez""","""25 of 50""","""50%""","""24 of 49""","""1 of 1""","""0 of 0""","""21 of 44""","""4 of 6""","""0 of 0""","""Gregory Rodrigues""","""58 of 95""","""61%""","""42 of 76""","""8 of 11""","""8 of 8""","""54 of 89""","""4 of 6""","""0 of 0""","""2026-09-03T12:54:37+00:00""","""2026-09-03T13:23:47+00:00""","""a0a69dc9914ef6e1""","""fb77b08a90b92d5b"""
"""Round 5""","""Anthony Hernandez""","""24 of 31""","""77%""","""24 of 31""","""0 of 0""","""0 of 0""","""16 of 20""","""5 of 8""","""3 of 3""","""Gregory Rodrigues""","""27 of 49""","""55%""","""18 of 40""","""6 of 6""","""3 of 3""","""21 of 41""","""6 of 8""","""0 of 0""","""2026-09-03T12:54:37+00:00""","""2026-09-03T13:23:47+00:00""","""a0a69dc9914ef6e1""","""fb77b08a90b92d5b"""


In [7]:
import polars as pl

CATEGORIES = ["sig_str", "head", "body", "leg", "distance", "clinch", "ground"]
METRICS = ["landed", "attempted", "pct"]
FIGHTERS = [1, 2]

BASE_COLS = [
    "fight_id",
    "event_id",
    "round",
    "scraped_at",
    "ingested_at",
]

In [8]:
def transform_significant_strikes(significant_strikes: pl.DataFrame) -> pl.DataFrame:
    exprs_values = []
    exprs_values_pct = []

    cols_to_drop = [
        f"{category}_fighter{fighter}"
        for category in CATEGORIES
        for fighter in FIGHTERS
    ]

    for category in CATEGORIES:
        for fighter in FIGHTERS:
            exprs_values.extend([
                pl.col(f"{category}_fighter{fighter}")
                .str.extract(r"(\d+) of (\d+)", 1)
                .cast(pl.Int64)
                .alias(f"{category}_landed_fighter{fighter}"),

                pl.col(f"{category}_fighter{fighter}")
                .str.extract(r"(\d+) of (\d+)", 2)
                .cast(pl.Int64)
                .alias(f"{category}_attempted_fighter{fighter}")
            ])

    for category in CATEGORIES:
        for fighter in FIGHTERS:
            exprs_values_pct.append(
                (
                    100
                    * pl.col(f"{category}_landed_fighter{fighter}")
                    / pl.col(f"{category}_attempted_fighter{fighter}")
                )
                .fill_nan(0)
                .fill_null(0)
                .round(2)
                .alias(f"{category}_pct_fighter{fighter}")
            )

    return (
        significant_strikes
        .drop("sig_str_pct_fighter1", "sig_str_pct_fighter2")
        .with_columns(*exprs_values)
        .with_columns(*exprs_values_pct)
        .drop(cols_to_drop)
        .with_columns(
            pl.col("round_label")
            .str.extract(r"(\d+)")
            .cast(pl.Int64)
            .alias("round"),

            pl.col("scraped_at")
            .str.strip_chars('"')
            .str.to_datetime(format="%Y-%m-%dT%H:%M:%S%z"),

            pl.col("ingested_at")
            .str.strip_chars('"')
            .str.to_datetime(format="%Y-%m-%dT%H:%M:%S%z"),

            pl.col("fighter1").str.strip_chars(),
            pl.col("fighter2").str.strip_chars(),
        )
        .drop("round_label")
    )

In [9]:
def build_fighter_view(df: pl.DataFrame, fighter_num: int) -> pl.DataFrame:
    other = 2 if fighter_num == 1 else 1

    exprs = [
        pl.col(f"fighter{fighter_num}").alias("fighter"),
        pl.col(f"fighter{other}").alias("opponent"),
    ]

    for category in CATEGORIES:
        for metric in METRICS:
            exprs.append(
                pl.col(f"{category}_{metric}_fighter{fighter_num}")
                .alias(f"{category}_{metric}")
            )

    return df.select(*BASE_COLS, *exprs)

In [10]:
import polars as pl


def create_significant_strikes_long(
    significant_strikes: pl.DataFrame,
) -> pl.DataFrame:

    fighter1_df = build_fighter_view(significant_strikes, 1)
    fighter2_df = build_fighter_view(significant_strikes, 2)

    return pl.concat(
        [fighter1_df, fighter2_df],
        how="vertical",
    )

In [11]:
significant_strikes_long = (
    significant_strikes
    .pipe(transform_significant_strikes)
    .pipe(create_significant_strikes_long)
)

In [ ]:
# significant_strikes_clean = transform_significant_strikes(
#     significant_strikes
# )

# significant_strikes_long = create_significant_strikes_long(
#     significant_strikes_clean
# )


In [12]:
significant_strikes_long

fight_id,event_id,round,scraped_at,ingested_at,fighter,opponent,sig_str_landed,sig_str_attempted,sig_str_pct,head_landed,head_attempted,head_pct,body_landed,body_attempted,body_pct,leg_landed,leg_attempted,leg_pct,distance_landed,distance_attempted,distance_pct,clinch_landed,clinch_attempted,clinch_pct,ground_landed,ground_attempted,ground_pct
str,str,i64,"datetime[μs, UTC]","datetime[μs, UTC]",str,str,i64,i64,f64,i64,i64,f64,i64,i64,f64,i64,i64,f64,i64,i64,f64,i64,i64,f64,i64,i64,f64
"""fb77b08a90b92d5b""","""a0a69dc9914ef6e1""",1,2026-09-03 12:54:37 UTC,2026-09-03 13:23:47 UTC,"""Anthony Hernandez""","""Gregory Rodrigues""",10,19,52.63,10,19,52.63,0,0,0.0,0,0,0.0,5,12,41.67,5,7,71.43,0,0,0.0
"""fb77b08a90b92d5b""","""a0a69dc9914ef6e1""",2,2026-09-03 12:54:37 UTC,2026-09-03 13:23:47 UTC,"""Anthony Hernandez""","""Gregory Rodrigues""",19,39,48.72,19,39,48.72,0,0,0.0,0,0,0.0,18,37,48.65,1,2,50.0,0,0,0.0
"""fb77b08a90b92d5b""","""a0a69dc9914ef6e1""",3,2026-09-03 12:54:37 UTC,2026-09-03 13:23:47 UTC,"""Anthony Hernandez""","""Gregory Rodrigues""",12,20,60.0,12,20,60.0,0,0,0.0,0,0,0.0,4,12,33.33,5,5,100.0,3,3,100.0
"""fb77b08a90b92d5b""","""a0a69dc9914ef6e1""",4,2026-09-03 12:54:37 UTC,2026-09-03 13:23:47 UTC,"""Anthony Hernandez""","""Gregory Rodrigues""",25,50,50.0,24,49,48.98,1,1,100.0,0,0,0.0,21,44,47.73,4,6,66.67,0,0,0.0
"""fb77b08a90b92d5b""","""a0a69dc9914ef6e1""",5,2026-09-03 12:54:37 UTC,2026-09-03 13:23:47 UTC,"""Anthony Hernandez""","""Gregory Rodrigues""",24,31,77.42,24,31,77.42,0,0,0.0,0,0,0.0,16,20,80.0,5,8,62.5,3,3,100.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""116dc88b63113f61""","""a0a69dc9914ef6e1""",3,2026-09-03 12:56:29 UTC,2026-09-03 13:23:47 UTC,"""Stan Dorsainvil""","""Gauge Young""",50,101,49.5,45,92,48.91,4,7,57.14,1,2,50.0,33,79,41.77,17,22,77.27,0,0,0.0
"""11e4a448ce1ae5e3""","""a0a69dc9914ef6e1""",1,2026-09-03 12:56:40 UTC,2026-09-03 13:23:47 UTC,"""Jackson McVey""","""Wes Schultz""",38,78,48.72,32,69,46.38,6,9,66.67,0,0,0.0,34,72,47.22,1,3,33.33,3,3,100.0
"""97fb716bc7b8751a""","""a0a69dc9914ef6e1""",1,2026-09-03 12:56:51 UTC,2026-09-03 13:23:47 UTC,"""Elise Reed""","""Shanelle Dyer""",19,40,47.5,3,19,15.79,10,11,90.91,6,10,60.0,17,37,45.95,2,3,66.67,0,0,0.0


In [13]:
significant_strikes_long.shape

(52, 28)

In [21]:
len(significant_strikes_long["fight_id"].unique().to_list())

13